In [ ]:
import astropy.table as at
import jax
import matplotlib.pyplot as plt
import numpy as np
import harv
import unxt as u
import quaxed.numpy as jnp

jax.config.update("jax_enable_x64", True)

# Multi-survey offsets

In [ ]:
rng = np.random.default_rng(1234)
times1 = rng.uniform(0, 100, 16)
times2 = rng.uniform(50, 150, 12)
times = np.concatenate([times1, times2])

mask = np.zeros(len(times), dtype=bool)
mask[len(times1) :] = True

truth = {
    "P": 24.5234,
    "e": 0.1833,
    "t_peri": 4.139,
    "arg_peri": 51.394,
    "rv_semiamp": 1.3523,
    "v_sys": 40.0,
}
rv = harv.kepler.rv_at_times(
    u.Q(times, "day"),
    u.Q(truth["P"], "day"),
    truth["e"],
    t_peri=u.Q(truth["t_peri"], "day"),
    arg_peri=u.Q(truth["arg_peri"], "deg"),
    rv_semiamp=u.Q(truth["rv_semiamp"], "km/s"),
    v_sys=u.Q(truth["v_sys"], "km/s"),
)
rv = rv.at[mask].set(rv[mask] + u.Q(0.4, "km/s"))

rv_err = u.Q(rng.uniform(0.08, 0.16, len(times)), "km/s")
rv = rv + u.Q(rng.normal(0, rv_err.value), "km/s")

In [ ]:
plt.errorbar(times1, rv[~mask], yerr=rv_err[~mask], fmt="o")
plt.errorbar(times2, rv[mask], yerr=rv_err[mask], fmt="o")

In [ ]:
tbl = at.Table({"time": times, "rv": rv, "rv_err": rv_err})
survey = np.full(len(tbl), "survey1", dtype="U7")
survey[mask] = "survey2"
tbl["survey"] = survey
for k, v in truth.items():
    tbl.meta[k] = v
tbl.write("simulated-multi-survey-data.fits", overwrite=True)